# Web Scraping Opinión - El Español
Scraper con paginación (hasta 20 páginas por sección) y límite de 2500 artículos.

In [1]:

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time


In [2]:

SECCIONES_BASE = [
    "https://www.elespanol.com/opinion/",
    "https://www.elespanol.com/opinion/carta-del-director/",
    "https://www.elespanol.com/opinion/editoriales/",
    "https://www.elespanol.com/opinion/columnas/",
    "https://www.elespanol.com/opinion/tribunas/",
    "https://www.elespanol.com/opinion/vinetas/",
    "https://www.elespanol.com/opinion/videoblog-director/",
]

MAX_PAGINAS = 20
MAX_ARTICULOS = 2500

HEADERS = {"User-Agent": "Mozilla/5.0"}


In [3]:

def get_soup(url):
    r = requests.get(url, headers=HEADERS, timeout=10)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")

def es_articulo(url):
    return "/opinion/" in url and url.count("/") > 4


In [4]:

urls = set()

for seccion in SECCIONES_BASE:
    print(f"Sección: {seccion}")

    for pagina in range(1, MAX_PAGINAS + 1):

        if len(urls) >= MAX_ARTICULOS:
            break

        if pagina == 1:
            url = seccion
        else:
            url = f"{seccion}{pagina}/"

        print(f"  Página {pagina}")

        try:
            soup = get_soup(url)
            enlaces = soup.select("a[href]")

            nuevos = 0

            for a in enlaces:
                href = a["href"]

                if href.startswith("https://") and es_articulo(href):
                    if href not in urls:
                        urls.add(href)
                        nuevos += 1

                if len(urls) >= MAX_ARTICULOS:
                    break

            print(f"    +{nuevos} nuevos (total {len(urls)})")

        except Exception as e:
            print("Error:", e)

        time.sleep(0.5)

    if len(urls) >= MAX_ARTICULOS:
        break

print(f"Total URLs: {len(urls)}")


Sección: https://www.elespanol.com/opinion/
  Página 1
    +29 nuevos (total 29)
  Página 2
    +29 nuevos (total 58)
  Página 3
    +29 nuevos (total 87)
  Página 4
    +29 nuevos (total 116)
  Página 5
    +29 nuevos (total 145)
  Página 6
    +29 nuevos (total 174)
  Página 7
    +30 nuevos (total 204)
  Página 8
    +29 nuevos (total 233)
  Página 9
    +29 nuevos (total 262)
  Página 10
    +28 nuevos (total 290)
  Página 11
    +29 nuevos (total 319)
  Página 12
    +29 nuevos (total 348)
  Página 13
    +30 nuevos (total 378)
  Página 14
    +30 nuevos (total 408)
  Página 15
    +30 nuevos (total 438)
  Página 16
    +29 nuevos (total 467)
  Página 17
    +29 nuevos (total 496)
  Página 18
    +29 nuevos (total 525)
  Página 19
    +29 nuevos (total 554)
  Página 20
    +29 nuevos (total 583)
Sección: https://www.elespanol.com/opinion/carta-del-director/
  Página 1
    +13 nuevos (total 596)
  Página 2
    +30 nuevos (total 626)
  Página 3
    +30 nuevos (total 656)
  Página 4


In [5]:

def scrape_article(url):
    soup = get_soup(url)

    title = soup.find("h1")
    title = title.get_text(strip=True) if title else ""

    paragraphs = soup.select("article p")
    text = " ".join(p.get_text(strip=True) for p in paragraphs)

    return {
        "periodico": "el español",
        "titulo": title.lower(),
        "texto": text.lower(),
        "url": url.lower()
    }

rows = []

for url in list(urls):

    try:
        art = scrape_article(url)

        if art["titulo"] and art["texto"]:
            rows.append(art)

        if len(rows) >= MAX_ARTICULOS:
            break

        time.sleep(0.5)

    except Exception as e:
        print("Error:", e)

print("Artículos scrapeados:", len(rows))


Error: HTTPSConnectionPool(host='www.elespanol.com', port=443): Read timed out.
Error: HTTPSConnectionPool(host='www.elespanol.com', port=443): Read timed out.
Error: HTTPSConnectionPool(host='www.elespanol.com', port=443): Read timed out.
Error: HTTPSConnectionPool(host='www.elespanol.com', port=443): Read timed out.
Error: HTTPSConnectionPool(host='www.elespanol.com', port=443): Read timed out.
Artículos scrapeados: 2192


In [6]:

df = pd.DataFrame(rows)
df.to_csv("elespanol_opinion.csv", index=False, encoding="utf-8-sig")
df.head()


,periodico,titulo,texto,url
0,el español,"albert rivera, entre escila y caribdis","a medida que transcurren los días, desde el fi...",https://www.elespanol.com/opinion/carta-del-di...
1,el español,la elección de isabel perelló demuestra que la...,reunión del pleno del consejo general del pode...,https://www.elespanol.com/opinion/editoriales/...
2,el español,por qué le hablas a tu novio como si fuese una...,"fotograma de alta fidelidad. el otro día, mi a...",https://www.elespanol.com/opinion/columnas/202...
3,el español,"una instrucción extemporánea, mal razonada y c...","carlos mazón, en su comparecencia en el congre...",https://www.elespanol.com/opinion/editoriales/...
4,el español,"montero maquilla la financiación: más dinero, ...",la ministra maría jesús montero.mariscalagenci...,https://www.elespanol.com/opinion/tribunas/202...
